# Understand and compare pancreas MRI segmentations

**Question:** can these pretrained models produce a usable pancreas mask on our two MRI
examples, and what changed after correcting the UniverSeg inputs?

Run the cells from top to bottom. This notebook reads saved scans and predictions;
**it does not download data, train a model, or run inference**. Missing prediction files
are reported in the table. See the README's MRI setup instructions if local files are absent.

A scan is a 3D grid of voxels. A **binary mask** has the same grid, with 1 marking pancreas
and 0 marking background. We compare each model's predicted mask with the dataset's
reference annotation, which is our evaluation target rather than a model input.

## 1. What is a support set, and what does UniverSeg do?

The **query** is the image we want to segment. A **support example** is an image from
another patient paired with its known mask. The **support set** is the collection of
these image–mask pairs: it shows the model what structure to find in the query.

UniverSeg is a pretrained **2D** segmentation network. It jointly processes a query slice
and labeled support slices, exchanging information between their learned features to
predict the query mask. The support masks specify the task (here, pancreas).
**We do not train or fine-tune its weights on these examples.** Changing the support set
changes the information supplied at prediction time, not the pretrained weights.

```text
Support MRI slices + their pancreas masks ─┐
                                         ├─> pretrained UniverSeg ─> query prediction
Query MRI slice (without its mask) ───────┘
Query reference mask ─────────────────────────────> scoring afterward only
```

Our script repeats this operation over the query volume's axial slices, using the same
support set, then restores the prediction to the scan's original grid. This is our
whole-volume adaptation of a 2D model.

Sources: [official model and input interface](https://github.com/JJGO/UniverSeg),
[ICCV 2023 paper](https://openaccess.thecvf.com/content/ICCV2023/papers/Butoi_UniverSeg_Universal_Medical_Image_Segmentation_ICCV_2023_paper.pdf).

## 2. What are we comparing?

All predictions below are evaluated on the **same two development queries**:
T1 `EMC014_T1` and T2 `CAD_T2_110`. T1 and T2 are different MRI acquisition sequences;
we use same-sequence support for each query.

| Saved run | Information available at inference | Purpose here |
| --- | --- | --- |
| TotalSegmentator | Query MRI; no support annotations | Automatic pretrained comparator; pancreas extracted from its multi-organ output |
| UniverSeg · original 4 | Query + 4 labeled slices from 1 separate patient; per-slice normalization | Preserve the initial smoke test |
| UniverSeg · updated 64 | Query + 64 labeled slices from 4 separate patients; corrected volume preprocessing | Inspect the combined input correction |

The updated pool contains **16 foreground-containing processed slices per patient**,
selected evenly across that patient's pancreas-containing axial range:

| Sequence | Original support patient | Updated support patients |
| --- | --- | --- |
| T1 | `EMC009_T1` | `EMC009_T1`, `EMC010_T1`, `EMC011_T1`, `EMC021_T1` |
| T2 | `CAD_T2_102` | `CAD_T2_102`, `CAD_T2_107`, `CAD_T2_113`, `CAD_T2_120` |

These are separate training-set patient IDs; neither query is in its support pool.
Support labels may guide support selection. Query labels are used only for evaluation
and choosing a display slice. **64 support slices means 64 image–mask pairs, not 64
patients.** Resampling can create several views between acquired slices; it adds no
new acquired anatomy.

## 3. Why preprocess, and what was checked?

MRI intensities have no fixed common scale, and the model expects 128 × 128 slices.
The updated run reorients each volume consistently, clips extreme intensities using the
0.5th/99.5th percentiles of nonzero voxels, and scales the whole volume to [0, 1].
It resizes to 128³ using smooth image interpolation and nearest-neighbor mask interpolation
so labels stay binary. After inference, probabilities are resized back to the original
grid before applying the 0.5 threshold. A cubic array does not imply isotropic physical voxels.

Before the updated inference, we checked image/mask grids, finite normalized intensities,
binary masks, and all 64 support overlays per sequence. Local inspection sheets:
[processed queries](../results/universeg-preprocessing-64/processed-query-check.png),
[T1 support](../results/universeg-preprocessing-64/T1-support-64.png),
[T2 support](../results/universeg-preprocessing-64/T2-support-64.png).
These links require the local run artifacts; they are not stored in Git.

The cells below check that each saved prediction and reference still share the original
scan grid before scoring. Matching array sizes alone would not establish alignment.

## 4. Load saved outputs and inspect the overlays

The paths below separate the original run from the corrected UniverSeg run. Nothing is
recomputed by a model. Cyan outlines show the reference; orange outlines show predictions.
All panels for a case use the same slice, chosen where its reference has the most pancreas
voxels. This makes inspection easier, but can hide errors on other slices.

**Dice uses the entire original 3D volume**, not just the displayed slice:

$$\mathrm{Dice} = \frac{2\,|P\cap R|}{|P|+|R|}$$

Here, P and R are predicted and reference foreground voxels. Dice is 1 for perfect overlap
and 0 for no overlap when either mask contains foreground. We report two empty masks as
undefined (`NaN`). Dice measures overlap; it does not establish clinical usefulness.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import nibabel as nib
import numpy as np
import pandas as pd

from ml_biomed_project.metrics import dice_score

project = Path.cwd().resolve()
if project.name == "notebooks":
    project = project.parent
original_run = project / "results" / "first-mri-smoke-test"
updated_run = project / "results" / "universeg-preprocessing-64"
export_run = project / "results" / "notebook-comparison"
cases = [("T1", "EMC014_T1"), ("T2", "CAD_T2_110")]
rows = []
figures = []

In [ ]:
for sequence, case in cases:
    folder = project / "data" / "crosspan" / sequence / "val"
    image = nib.load(folder / "images" / f"{case}_0000.nii.gz")
    reference = nib.load(folder / "labels" / f"{case}.nii.gz")
    if image.shape != reference.shape or not np.allclose(
        image.affine, reference.affine, rtol=0, atol=1e-5
    ):
        raise ValueError(f"Reference grid mismatch: {case}")
    data = image.get_fdata(dtype=np.float32)
    truth = np.asarray(reference.dataobj)
    if not np.all(np.isfinite(data)) or not np.all((truth == 0) | (truth == 1)):
        raise ValueError(f"Invalid image values or reference labels: {case}")
    if image.header.get_xyzt_units()[0] != reference.header.get_xyzt_units()[0]:
        raise ValueError(f"Spatial units mismatch: {case}")
    slice_index = int(np.argmax(np.count_nonzero(truth, axis=(0, 1))))
    spacing = image.header.get_zooms()
    limits = np.percentile(data, [1, 99])
    predictions = {"Reference": truth}
    for model, path in (
        ("TotalSegmentator", original_run / "totalsegmentator" / case / "multilabel.nii.gz"),
        ("UniverSeg · original 4", original_run / "universeg" / case / "pancreas.nii.gz"),
        ("UniverSeg · updated 64", updated_run / sequence / case / "pancreas.nii.gz"),
    ):
        if not path.exists():
            rows.append({"case": case, "model": model, "status": "missing output"})
            continue
        volume = nib.load(path)
        if volume.shape != image.shape or not np.allclose(
            volume.affine, image.affine, rtol=0, atol=1e-5
        ):
            raise ValueError(f"Prediction grid mismatch: {model} {case}")
        if volume.header.get_xyzt_units()[0] != image.header.get_xyzt_units()[0]:
            raise ValueError(f"Prediction spatial units mismatch: {model} {case}")
        values = np.asarray(volume.dataobj)
        if not np.all(np.isfinite(values)) or not np.all(values == np.floor(values)):
            raise ValueError(f"Invalid prediction labels: {model} {case}")
        if model == "TotalSegmentator":
            # Pancreas is class 7 in the pinned total_mr multilabel mapping.
            prediction = values == 7
        else:
            if not np.all((values == 0) | (values == 1)):
                raise ValueError(f"UniverSeg prediction must be binary: {case}")
            prediction = values.astype(bool)
        score = dice_score(prediction, truth)
        rows.append(
            {
                "case": case,
                "model": model,
                "status": "evaluated",
                "dice": score,
                "prediction_voxels": int(prediction.sum()),
                "reference_voxels": int(truth.sum()),
            }
        )
        predictions[model] = prediction

    fig, axes = plt.subplots(
        1, len(predictions), figsize=(5 * len(predictions), 4), layout="constrained"
    )
    for ax, (name, mask) in zip(np.atleast_1d(axes), predictions.items(), strict=True):
        ax.imshow(
            data[:, :, slice_index].T,
            cmap="gray",
            origin="lower",
            aspect=float(spacing[1] / spacing[0]),
            vmin=limits[0],
            vmax=limits[1],
        )
        reference_slice = truth[:, :, slice_index].T
        mask_slice = mask[:, :, slice_index].T
        if np.any(reference_slice):
            ax.contour(reference_slice, levels=[0.5], colors=["cyan"], linewidths=1)
        if name != "Reference" and np.any(mask_slice):
            ax.contour(mask_slice, levels=[0.5], colors=["orange"], linewidths=1)
        ax.set(title=name, xlabel="Array axis 0 (voxels)", ylabel="Array axis 1 (voxels)")
    fig.suptitle(f"{case} — slice {slice_index}; cyan: reference, orange: prediction")
    figures.append((case, fig))
    plt.show()

scores = pd.DataFrame(rows)
print(scores.to_string(index=False))

## 5. Read the results with the comparison in mind

Read each case separately: a higher Dice means better whole-volume overlap for that case.
The voxel counts help identify over- or under-segmentation, but correct counts alone do
not guarantee correct location. Compare the outlines as well as the score. Missing outputs
and undefined Dice remain explicit; inspect local run logs if an output is missing.

The two UniverSeg runs use the same checkpoint and queries, but preprocessing, support
count, and patient diversity changed together. Their difference describes the **combined
correction**; it cannot establish which change helped. TotalSegmentator receives no support
annotations, so this is also not an equal-information model ranking.

These are two previously inspected development cases, not an independent test cohort.
The T1 scan has only nine acquired slices at 23.2 mm spacing. Pretraining overlap remains
unverified. We use one support set per sequence, without the paper's support-set ensembling;
our whole-volume evaluation differs from its selected-slice benchmark. These observations
do not establish performance on other patients or MRI sequences.

## Optional: save this comparison locally

Run the next cell only if you want a CSV and the displayed figures under
`results/notebook-comparison/`. Rerunning it replaces those notebook exports; the original
model outputs remain in their separate run folders. Results stay local and ignored by Git.
Clear notebook outputs before committing.

In [ ]:
export_run.mkdir(parents=True, exist_ok=True)
scores.to_csv(export_run / "scores.csv", index=False)
for case, figure in figures:
    figure.savefig(export_run / f"{case}-overlay.png", dpi=140)
print(f"Saved results under {export_run}")